# Tune A Detection Rule Until The On Call Engineer Trusts It

Companion notebook for the [Detection Rules and Alert Fatigue](https://codemia.io/courses/security_engineering/sec_detection_rules_and_alert_fatigue) lesson on Codemia.

**What you will build**: a detection engineering lab on a synthetic authentication log built with pandas. The log mixes
ordinary users who mistype passwords, a shared office address, a misconfigured service account and a
few planted account takeovers. You will write the failures then success rule, sweep its threshold and
window to see precision and recall move against each other, find which benign behaviours cause the
noise, add suppression and enrichment, and price the result in pages per week.

## Setup

In [ ]:
%pip install -q pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
WEEK = 7 * 86_400
pd.set_option('display.width', 120)
print('pandas', pd.__version__)

## Concept recap

A detection rule is a hypothesis about what an attack looks like in your logs, and it is judged on
two numbers. Recall is the fraction of real attacks it catches. Precision is the fraction of its
alerts that are real. Attacks are rare, so even a rule with a tiny false positive rate per event can
produce mostly false alerts, and an analyst who sees mostly false alerts stops reading them. That is
alert fatigue, and it is how real intrusions get acknowledged and closed. This notebook tunes one
rule against a week of synthetic logs where the truth is known.

## Build a week of authentication logs

In [ ]:
rows = []
history = set()     # (user, ip) pairs with a successful login in the previous 30 days
def login(t, user, ip, ok, label='benign', kind='user'):
    rows.append((int(t), user, ip, bool(ok), label, kind))

# 2,000 ordinary users: a few logins a day, occasional typos followed by a success
for u in range(2_000):
    home_ip = f'10.{u % 250}.{u // 250}.7'
    history.add((f'user{u}', home_ip))
    phone_ip = f'100.64.{u % 250}.{u // 250}'
    has_phone = rng.random() < 0.3        # some people also log in from a phone network
    if has_phone:
        history.add((f'user{u}', phone_ip))
    for t in np.sort(rng.uniform(0, WEEK, rng.poisson(14))):
        # mostly home, sometimes the phone, and now and then a hotel or a brand new device
        ip = rng.choice([home_ip, phone_ip, f'192.0.2.{rng.integers(1, 254)}'], p=[0.875, 0.12, 0.005] if has_phone else [0.995, 0.0, 0.005])
        typos = rng.choice([0, 1, 2, 3, 5], p=[0.82, 0.10, 0.04, 0.03, 0.01])
        if ip.startswith('192.0.2.'):
            typos = rng.choice([0, 2, 4, 7], p=[0.6, 0.2, 0.15, 0.05])   # new keyboard, forgotten password
        for k in range(typos):
            login(t + k * 8, f'user{u}', ip, False)
        login(t + typos * 8 + 5, f'user{u}', ip, True)

# an office behind one NAT address: 150 people, morning logins in a burst
for day in range(5):
    for u in range(150):
        t = day * 86_400 + 9 * 3600 + rng.normal(0, 600)
        for k in range(rng.choice([0, 1, 4], p=[0.85, 0.1, 0.05])):
            login(t + k * 6, f'office{u}', '203.0.113.10', False, kind='office')
        history.add((f'office{u}', '203.0.113.10'))
        login(t + 30, f'office{u}', '203.0.113.10', True, kind='office')

# a service account with a stale password in one config: fails every minute, the healthy pod succeeds
for t in range(0, WEEK, 60):
    login(t, 'svc-reporting', '10.9.9.9', False, kind='service')
    if t % 600 == 0:
        login(t + 2, 'svc-reporting', '10.9.9.8', True, kind='service')

# eight planted takeovers from new addresses: (failures before the success, seconds between tries).
# Fast scripted runs, patient ones that space out tries to stay under a threshold, and short ones
# where the attacker already had a good guess from a breach of another site.
PLAN = [(30, 2), (15, 2), (9, 20), (6, 20), (4, 5), (12, 90), (20, 240), (5, 600)]
attacks = []
for i, (n_fail, gap) in enumerate(PLAN):
    victim = f'user{rng.integers(2_000)}'
    t0 = rng.uniform(0, WEEK - n_fail * gap - 60)
    ip = f'198.51.100.{i + 20}'
    for k in range(n_fail):
        login(t0 + k * gap, victim, ip, False, 'attack', 'attack')
    login(t0 + n_fail * gap + 3, victim, ip, True, 'attack', 'attack')
    attacks.append((victim, ip, n_fail, gap))

log = pd.DataFrame(rows, columns=['t', 'user', 'ip', 'ok', 'label', 'kind']).sort_values('t', ignore_index=True)
print(f'{len(log):,} events, {(~log.ok).sum():,} failures, {log.user.nunique():,} accounts')
print(pd.DataFrame(attacks, columns=['victim', 'ip', 'failures', 'seconds between tries']))

## The rule: a success after N failures within a window

In [ ]:
def detect(log, threshold, window, key=('user', 'ip')):
    """Alert on a successful login when the same key saw at least `threshold` failures in the
    preceding `window` seconds. Returns one row per alerting success."""
    alerts = []
    for k, g in log.groupby(list(key), sort=False):
        fails = g.t[~g.ok].to_numpy()
        for t, label, kind in g.loc[g.ok, ['t', 'label', 'kind']].itertuples(index=False):
            n = np.searchsorted(fails, t) - np.searchsorted(fails, t - window)
            if n >= threshold:
                alerts.append((t, *((k,) if isinstance(k, str) else k), n, label, kind))
    cols = ['t', *key, 'failures', 'label', 'kind']
    return pd.DataFrame(alerts, columns=cols)

def score(alerts):
    tp = (alerts.label == 'attack').sum()
    caught = alerts.loc[alerts.label == 'attack', 'user'].nunique()
    return {'alerts': len(alerts), 'true': int(tp), 'precision': round(float(tp / max(len(alerts), 1)), 3),
            'recall': round(caught / len(attacks), 3)}

a = detect(log, threshold=3, window=300)
print(score(a))
print(a.groupby('kind').size().rename('alerts by source'))

The first version catches some attacks and drowns them. Read the breakdown by source before touching
any threshold: ordinary users with a few typos and the shared office address dominate. Those are not
attacks, and no single number separates every one of them from an attacker who tries a handful of
passwords, which is what the sweep below makes concrete.

## Sweep threshold and window

In [ ]:
grid = []
for window in (60, 300, 900, 3600):
    for threshold in (2, 3, 5, 8, 12):
        s = score(detect(log, threshold, window))
        grid.append({'window': window, 'threshold': threshold, **s})
grid = pd.DataFrame(grid)
print(grid.pivot(index='threshold', columns='window', values='precision').round(2).to_string())
print()
print(grid.pivot(index='threshold', columns='window', values='recall').round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for window, g in grid.groupby('window'):
    ax.plot(g.recall, g.precision, marker='o', label=f'window {window}s')
    for _, r in g.iterrows():
        ax.annotate(int(r.threshold), (r.recall, r.precision), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('recall (attacks caught)')
ax.set_ylabel('precision (alerts that are real)')
ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.set_title('Every threshold is a trade; labels are N')
plt.tight_layout()
plt.show()

Raising the threshold buys precision and loses the attackers who stop early. Widening the window
catches the slow attackers who space out their tries and lets more typo bursts and office mornings
in. There is no setting that wins on both, which is the lesson's point: a threshold rule alone cannot
use what an analyst would use to tell these apart.

## Suppression and enrichment

In [ ]:
# What an analyst knows that the rule does not:
# 1. the account's usual addresses: an alert from a known address is much less interesting
# 2. known shared or service sources, maintained as an allow list with an owner and an expiry
# 3. one alert per account per hour is enough; repeats add nothing
SHARED = {'203.0.113.10': 'office NAT, owner it-ops, review 2027-01'}

def enrich_and_suppress(alerts, suppress=3600):
    a = alerts.copy()
    a['new_ip'] = [(u, ip) not in history for u, ip in zip(a.user, a.ip)]
    a = a[a.new_ip & ~a.ip.isin(SHARED)]
    keep, last = [], {}
    for r in a.sort_values('t').itertuples():
        if r.t - last.get(r.user, -np.inf) >= suppress:
            keep.append(r.Index)
            last[r.user] = r.t
    return a.loc[keep]

for window, threshold in ((300, 3), (900, 5), (3600, 5)):
    raw = detect(log, threshold, window)
    tuned = enrich_and_suppress(raw)
    print(f'window {window:>4}s threshold {threshold}: raw {score(raw)}')
    print(f'{"":>26}tuned {score(tuned)}')

Requiring the success to come from an address the account has never logged in from before removes
almost all the typo noise, because people mistype from their own laptop. The shared office address is
listed with an owner and a review date, so the exception is visible and expires instead of becoming a
permanent blind spot. Suppression collapses repeats into one alert per account per hour. With those
three, a low threshold and a wide window are affordable, so recall comes back without the noise.

What is left is mostly real people on a genuinely new device or network who also fumbled their
password, and from the log alone they look exactly like an attacker. That residue is where a step up
challenge, a second factor prompt at login, beats an alert: the user clears it in seconds, and the
attacker cannot.

The service account never alerts at all, because its failures and successes come from different
addresses. That is a separate detection, a health alert routed to its owner, and it is a good example
of noise that should be fixed at the source rather than filtered.

## Price it in pages

In [ ]:
tuned = enrich_and_suppress(detect(log, 3, 3600))
raw = detect(log, 3, 300)
MINUTES_PER_TRIAGE = 12
for name, a in (('first rule', raw), ('tuned rule', tuned)):
    s = score(a)
    print(f'{name:<11} {s["alerts"]:>5} alerts/week, {s["alerts"] * MINUTES_PER_TRIAGE / 60:>6.1f} analyst hours, '
          f'{s["true"]:>3} real, recall {s["recall"]:.0%}, precision {s["precision"]:.1%}')

missed = set(v for v, *_ in attacks) - set(tuned.loc[tuned.label == 'attack', 'user'])
print('attacks the tuned rule still misses:', missed or 'none')

A rule is done when its alert volume fits the team and most alerts are worth opening. Write down the
attacks it still misses and cover them deliberately, with another rule or a preventive control such as
rate limiting and multi factor authentication, rather than by lowering the threshold until the queue
fills up again. Then keep the planted attacks as test cases, so the next change to the rule is checked
against them before it ships.

## Exercises

1. Add a password spraying attacker who tries two passwords against each of 300 accounts from one
   address. Does the per account rule see it? Write a rule keyed on `ip` that does.
2. Add impossible travel enrichment: give each user a home region and flag a success from a different
   region within an hour of a success at home. How does precision change?
3. Make the shared office allow list expire mid week and rerun. How many alerts come back, and who
   should be paged to renew it?
4. Turn the eight planted attacks into a regression test that fails if recall drops below 100 percent
   or alerts exceed 50 a week, and use it to evaluate a change of your own.

## What to take away

- Measure a rule with precision and recall against known attacks, and price it in analyst hours.
- Thresholds trade one for the other. Context is what improves both: new address, known shared
  sources, account history.
- Every allow list entry needs an owner and an expiry, or it becomes a permanent blind spot.
- Suppress repeats so one incident is one alert.
- Fix noisy sources at the source, and cover known misses deliberately with another rule or a control.
- Keep planted attacks as regression tests for every rule change.